In [0]:
# COMMAND -----------
# Notebook: 03_gold_aggregations
# Description: Aggregate Silver layer data into Gold business metrics and KPIs.

import pyspark.sql.functions as F

# COMMAND -----------
# Step 1: Read cleaned data from Silver layer
df_silver = spark.table("ecommerce_silver.online_retail")

# Filter out non-cancelled orders for revenue calculations
df_sales = df_silver.filter(F.col("IsCancelled") == False)

# COMMAND -----------
# Gold Table 1: Monthly Revenue & Performance KPIs
# Aggregates revenue, order volume, and distinct customers by month
df_gold_monthly_revenue = (
    df_sales
    .withColumn("YearMonth", F.date_format("InvoiceDate", "yyyy-MM"))
    .groupBy("YearMonth")
    .agg(
        F.round(F.sum("TotalAmount"), 2).alias("TotalRevenue"),
        F.countDistinct("InvoiceNo").alias("TotalOrders"),
        F.countDistinct("CustomerID").alias("ActiveCustomers"),
        F.sum("Quantity").alias("TotalUnitsSold"),
        F.round(F.avg("TotalAmount"), 2).alias("AvgLineItemValue")
    )
    .orderBy("YearMonth")
)

# Write Monthly Revenue Table
(
    df_gold_monthly_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_gold.monthly_revenue")
)

print("Gold table created: ecommerce_gold.monthly_revenue")

# COMMAND -----------
# Gold Table 2: Customer Lifetime Value & RFM Base Metrics
# Aggregates customer behavior metrics (Total Spend, Order Count, Recency)
df_gold_customer_kpis = (
    df_sales
    .filter(F.col("CustomerID").isNotNull())
    .groupBy("CustomerID")
    .agg(
        F.countDistinct("InvoiceNo").alias("TotalOrders"),
        F.round(F.sum("TotalAmount"), 2).alias("TotalLifetimeSpend"),
        F.round(F.avg("TotalAmount"), 2).alias("AvgOrderValue"),
        F.min("InvoiceDate").alias("FirstPurchaseDate"),
        F.max("InvoiceDate").alias("LatestPurchaseDate"),
        F.first("Country").alias("PrimaryCountry")
    )
    .orderBy(F.col("TotalLifetimeSpend").desc())
)

# Write Customer KPIs Table
(
    df_gold_customer_kpis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_gold.customer_kpis")
)

print("Gold table created: ecommerce_gold.customer_kpis")

# COMMAND -----------
# Gold Table 3: Top Products Analysis
# Ranks products by total units sold and generated revenue
df_gold_top_products = (
    df_sales
    .groupBy("StockCode", "Description")
    .agg(
        F.sum("Quantity").alias("TotalUnitsSold"),
        F.round(F.sum("TotalAmount"), 2).alias("TotalRevenueGenerated"),
        F.countDistinct("InvoiceNo").alias("OrderCount")
    )
    .orderBy(F.col("TotalRevenueGenerated").desc())
)

# Write Top Products Table
(
    df_gold_top_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_gold.top_products")
)

print("Gold table created: ecommerce_gold.top_products")

# COMMAND -----------
# Display previews of all 3 Gold tables
print("=== Monthly Revenue Summary ===")
display(spark.table("ecommerce_gold.monthly_revenue").limit(5))

print("=== Top Customer KPIs ===")
display(spark.table("ecommerce_gold.customer_kpis").limit(5))

print("=== Top Performing Products ===")
display(spark.table("ecommerce_gold.top_products").limit(5))

Gold table created: ecommerce_gold.monthly_revenue
Gold table created: ecommerce_gold.customer_kpis
Gold table created: ecommerce_gold.top_products
=== Monthly Revenue Summary ===


YearMonth,TotalRevenue,TotalOrders,ActiveCustomers,TotalUnitsSold,AvgLineItemValue
2010-12,821452.73,1559,885,358019,20.04
2011-01,689811.61,1086,741,387099,20.25
2011-02,522545.56,1100,758,282934,19.44
2011-03,716215.26,1454,974,376599,20.18
2011-04,536968.49,1246,856,307953,18.59


=== Top Customer KPIs ===


CustomerID,TotalOrders,TotalLifetimeSpend,AvgOrderValue,FirstPurchaseDate,LatestPurchaseDate,PrimaryCountry
14646,73,280206.02,134.97,2010-12-20,2011-12-08,Netherlands
18102,60,259657.3,602.45,2010-12-07,2011-12-09,United Kingdom
17450,46,194390.79,578.54,2010-12-07,2011-12-01,United Kingdom
16446,2,168472.5,56157.5,2011-05-18,2011-12-09,United Kingdom
14911,201,143711.17,25.35,2010-12-01,2011-12-08,EIRE


=== Top Performing Products ===


StockCode,Description,TotalUnitsSold,TotalRevenueGenerated,OrderCount
DOT,DOTCOM POSTAGE,706,206248.77,706
22423,REGENCY CAKESTAND 3 TIER,13851,174156.54,1988
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6,1
85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,104284.24,2189
47566,PARTY BUNTING,18283,99445.23,1685
